# Guía de formato de Board y Move

Lo más importante en un bot de ajedrez es la eficiencia (además de la correctitud obviamente). Para esto representaremos el tablero y los movimientos de una manera especial: usando bits. En particular, a lo largo de este proyecto representaremos los tableros usando _bitboards_, es decir, enteros de 64-bits tales que cada bit representa una casilla del tablero y el valor de cada bit denota la presencia o ausencia de una pieza en esa casilla. En general, trataremos de representar todo lo que podamos como bits para ahorrar tiempo y memoria.

Veamos unos ejemplos:

In [12]:
def print_board(board: int):
    bitboard = format(board, 'b').zfill(64)

    print('    A B C D E F G H')
    for rank in range(8):
        print(8 - rank, '|', ' '.join(bitboard[8 * rank:8 * (rank + 1)]))

tablero = 42
print('Tablero en forma binaria:', format(tablero, 'b'))
print('Tablero en forma binaria (llenado con ceros):\n' + format(tablero, 'b').zfill(64))
print()
print("Interpretación del tablero:")
print_board(tablero)

Tablero en forma binaria: 101010
Tablero en forma binaria (llenado con ceros):
0000000000000000000000000000000000000000000000000000000000101010

Interpretación del tablero:
    A B C D E F G H
8 | 0 0 0 0 0 0 0 0
7 | 0 0 0 0 0 0 0 0
6 | 0 0 0 0 0 0 0 0
5 | 0 0 0 0 0 0 0 0
4 | 0 0 0 0 0 0 0 0
3 | 0 0 0 0 0 0 0 0
2 | 0 0 0 0 0 0 0 0
1 | 0 0 1 0 1 0 1 0


Como pueden ver, el entero 42 representa un tablero con piezas en las posiciones C1, E1 y G1. Al transformar el entero a bits, su bit menos significativo es la esquina inferior derecha y su bit más significativo es la esquina superior izquierda. Veamos un ejemplo más extremo:

In [13]:
tablero_inicial = 18446462598732906495
print_board(tablero_inicial)

    A B C D E F G H
8 | 1 1 1 1 1 1 1 1
7 | 1 1 1 1 1 1 1 1
6 | 0 0 0 0 0 0 0 0
5 | 0 0 0 0 0 0 0 0
4 | 0 0 0 0 0 0 0 0
3 | 0 0 0 0 0 0 0 0
2 | 1 1 1 1 1 1 1 1
1 | 1 1 1 1 1 1 1 1


El entero 18446462598732906495 representa un tablero con piezas en todas las posiciones iniciales. Ahora se preguntarán, ¿de qué sirve esto si no nos sirve para diferenciar tipos de piezas? ¿Qué ventaja tiene usar bitboards en vez de listas?

Hay varias respuestas para esto. Primero, es mucho más eficiente en memoria guardar ints que listas. Segundo, en realidad basta con tener un bitboard para cada tipo de pieza de cada color para tener la información de todas las piezas. Tercero, y lo más importante, es que las operaciones entre bits (_bitwise operations_) son casi instantáneas. Esto nos servirá para hacer una variedad de operaciones muy rápidamente. Veamos un ejemplo de un tipo de operación que podemos hacer:

In [14]:
# Tablero que representa las posiciones con piezas negras
piezas_negras = 9953667836118302848
print("Piezas negras:")
print_board(piezas_negras)
print()

# Supongamos que tenemos una caballo en A4 y tenemos un tablero 
# que corresponde a las posiciones atacadas por nuestro caballo
tablero_ataque = 70506185244672
print("Casillas atacadas por caballo en A4:")
print_board(tablero_ataque)
print()

# Ahora, para saber qué piezas negras están siendo atacadas por nuestro caballo,
# basta con intersectar ambos bitboards para obtener el resultado
print("Piezas negras atacadas por caballo en A4:")
print_board(piezas_negras & tablero_ataque)

Piezas negras:
    A B C D E F G H
8 | 1 0 0 0 1 0 1 0
7 | 0 0 1 0 0 0 1 0
6 | 1 0 0 0 1 0 0 0
5 | 0 0 1 0 1 0 0 1
4 | 0 0 0 0 0 0 0 0
3 | 0 0 0 1 0 0 0 0
2 | 0 0 0 0 0 0 0 0
1 | 1 0 0 0 0 0 0 0

Casillas atacadas por caballo en A4:
    A B C D E F G H
8 | 0 0 0 0 0 0 0 0
7 | 0 0 0 0 0 0 0 0
6 | 0 1 0 0 0 0 0 0
5 | 0 0 1 0 0 0 0 0
4 | 0 0 0 0 0 0 0 0
3 | 0 0 1 0 0 0 0 0
2 | 0 1 0 0 0 0 0 0
1 | 0 0 0 0 0 0 0 0

Piezas negras atacadas por caballo en A4:
    A B C D E F G H
8 | 0 0 0 0 0 0 0 0
7 | 0 0 0 0 0 0 0 0
6 | 0 0 0 0 0 0 0 0
5 | 0 0 1 0 0 0 0 0
4 | 0 0 0 0 0 0 0 0
3 | 0 0 0 0 0 0 0 0
2 | 0 0 0 0 0 0 0 0
1 | 0 0 0 0 0 0 0 0


Como vemos, podemos obtener fácilmente las piezas atacadas por el caballo en A4 con una intersección casi instantánea en la CPU. Hay muchísimas operaciones que se simplifican usando _bitwise operations_, además de que son muchísimo más rápidas.

Los bitboards no son lo único que se puede representar como bits. Entre otras cosas, podemos representar el turno como un entero de 1 bit (0 = blancas, 1 = negras), o podemos representar los derechos de enroque como un número de 4 bits (formato KQkq, donde KQ son los enroques hacia el lado del rey y de la reina de las blancas, y kq de las negras. Si un bit vale 1, el jugador respectivo aún puede enrocar en esa dirección).

Recuerden que las operaciones de bits son intersección (AND `&`), unión (OR `|`), disyunción exclusiva (XOR `^`), SHIFT LEFT (`<<`) y SHIFT RIGHT (`>>`). Algunos ejemplos de sus usos:

In [15]:
# Revisar si hay una pieza negra en la casilla X
# Supongamos X = 17
posicion = 1 << 17 # Setteamos el bit 17
print("Pieza negra en posición 17:")
print_board(piezas_negras & posicion) # Intersectamos las piezas negras con la posicion 17
print()
# X = 7
posicion = 1 << 7 # Setteamos el bit 7
print("Pieza negra en posición 7:")
print_board(piezas_negras & posicion) # Intersectamos las piezas negras con la posicion 7

# Podemos usar esto como un bool
if piezas_negras & (1 << 7):
    print("¡Hay una pieza en la posición 7 (A1)!")
print()

# Cambiar el turno después de cada movimiento
turno = 0
for _ in range(7):
    print(f"Turno: {"Negras" if turno else "Blancas"}")
    turno ^= 1 # Al hacerle XOR con 1 se invierte el bit. 1 XOR 1 = 0, 0 XOR 1 = 1.
print()

# Conocer todas las piezas del tablero a partir de las blancas y las negras
piezas_iniciales_negras = 18446462598732840960
piezas_iniciales_blancas = 65535
print("Piezas negras:")
print_board(piezas_iniciales_negras)
print()
print("Piezas blancas:")
print_board(piezas_iniciales_blancas)
print()
print("Piezas negras o blancas:")
print_board(piezas_iniciales_negras | piezas_iniciales_blancas) # Unimos ambos tableros
print()

Pieza negra en posición 17:
    A B C D E F G H
8 | 0 0 0 0 0 0 0 0
7 | 0 0 0 0 0 0 0 0
6 | 0 0 0 0 0 0 0 0
5 | 0 0 0 0 0 0 0 0
4 | 0 0 0 0 0 0 0 0
3 | 0 0 0 0 0 0 0 0
2 | 0 0 0 0 0 0 0 0
1 | 0 0 0 0 0 0 0 0

Pieza negra en posición 7:
    A B C D E F G H
8 | 0 0 0 0 0 0 0 0
7 | 0 0 0 0 0 0 0 0
6 | 0 0 0 0 0 0 0 0
5 | 0 0 0 0 0 0 0 0
4 | 0 0 0 0 0 0 0 0
3 | 0 0 0 0 0 0 0 0
2 | 0 0 0 0 0 0 0 0
1 | 1 0 0 0 0 0 0 0
¡Hay una pieza en la posición 7 (A1)!

Turno: Blancas
Turno: Negras
Turno: Blancas
Turno: Negras
Turno: Blancas
Turno: Negras
Turno: Blancas

Piezas negras:
    A B C D E F G H
8 | 1 1 1 1 1 1 1 1
7 | 1 1 1 1 1 1 1 1
6 | 0 0 0 0 0 0 0 0
5 | 0 0 0 0 0 0 0 0
4 | 0 0 0 0 0 0 0 0
3 | 0 0 0 0 0 0 0 0
2 | 0 0 0 0 0 0 0 0
1 | 0 0 0 0 0 0 0 0

Piezas blancas:
    A B C D E F G H
8 | 0 0 0 0 0 0 0 0
7 | 0 0 0 0 0 0 0 0
6 | 0 0 0 0 0 0 0 0
5 | 0 0 0 0 0 0 0 0
4 | 0 0 0 0 0 0 0 0
3 | 0 0 0 0 0 0 0 0
2 | 1 1 1 1 1 1 1 1
1 | 1 1 1 1 1 1 1 1

Piezas negras o blancas:
    A B C D E F G H
8 | 

Ahora que ya entienden cómo funcionan los bits, les explicaré el formato de las dos informaciones más importantes que guardaremos como bits...

#### Tablero

El tablero es representado como una colección de 13 bitboards: 1 por cada pieza y color (12 total) y 1 que representa todas las piezas en el tablero. Adicionalmente, se podría guardar 2 bitboards que representaran todas las piezas por color.

Esta es una de las mejores maneras de mantenerlos separados para poder hacer las operaciones adecuadas. Es importante mantener todos los bitboards actualizados al hacer movimientos, ya que ciertos movimientos pueden afectar más de un bitboard.

#### Movimientos

Cada vez que queremos hacer un movimiento (con la función `make_move()` de la que nos encargaremos en el futuro), queremos guardar la información de este con el menor peso posible. Para esto, empaquetamos la información en un entero de 32 bits donde se distribuye de la siguiente manera:

- Bits 0-5: casilla de origen (0-63)
- Bits 6-11: casilla de destino (0-63)
- Bits 12-15: tipo de pieza (0-11 según el índice del bitboard)
- Bits 16-19: pieza capturada (0-11, o 15 = ninguna)
- Bits 20-21: tipo de movimiento (0 = normal, 1 = enroque, 2 = en passant, 3 = promoción)
- Bits 22-23: promoción (0 = reina, 1 = torre, 2 = alfil, 3 = caballo)

Si queremos obtener la información de un bit específico X, simplemente hacemos `move & (1 << X)`, lo cual dará `True` si el bit está activado y `False` en caso contrario.

Por último, ¿por qué queremos que las operaciones sean tan eficientes? Esto es porque, al buscar el mejor movimiento posible, el bot analizará millones de jugadas posibles. Mientras más profundo queremos que analice, más jugadas tendrá que generar y analizar. Por lo tanto, queremos que la generación y ejecución de movimientos sea lo más breve posible para que el bot no tome demasiado tiempo pensando.

Tips adicionales:

In [15]:
# Obtener los primeros X bits de un número N
# p.ej. X = 3
X = 3
primeros_X = ((1 << X) - 1)

N = 62 # 111110
print(format(N, 'b'), format(primeros_X, 'b'), format(N & primeros_X, 'b'))


# Obtener los bits de N con índices en [a, b]
a = 2
b = 6
N = 853
bits_a_b = (N >> a) & ((1 << b - a + 1) - 1) # Máscara de primeros bits en el rango
print(format(N, 'b'), format(bits_a_b, 'b'))

111110 111 110
1101010101 10101
